In [2]:
# Import system modules
import sys
import os
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

print(f"✓ Project Root: {project_root}")
print(f"✓ Python Version: {sys.version}")

✓ Project Root: c:\RAG
✓ Python Version: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]


## Step 1: Test Core Dependencies

In [3]:
# Test all required imports
print("Testing core dependencies...\n")

try:
    import qdrant_client
    version = getattr(qdrant_client, '__version__', None)
    if version is None:
        import importlib.metadata
        version = importlib.metadata.version("qdrant-client")
    print(f"✓ qdrant-client: {version}")
except Exception as e:
    print(f"❌ qdrant-client: {e}")

try:
    import langchain
    print(f"✓ langchain: {langchain.__version__}")
except Exception as e:
    print(f"❌ langchain: {e}")

try:
    import openai
    print(f"✓ openai: {openai.__version__}")
except Exception as e:
    print(f"❌ openai: {e}")

try:
    import numpy as np
    print(f"✓ numpy: {np.__version__}")
except Exception as e:
    print(f"❌ numpy: {e}")

try:
    import pandas as pd
    print(f"✓ pandas: {pd.__version__}")
except Exception as e:
    print(f"❌ pandas: {e}")

try:
    from dotenv import load_dotenv
    print(f"✓ python-dotenv: installed")
except Exception as e:
    print(f"❌ python-dotenv: {e}")


Testing core dependencies...

✓ qdrant-client: 1.17.0
✓ langchain: 1.2.10
✓ openai: 2.24.0
✓ numpy: 2.4.2
✓ pandas: 3.0.1
✓ python-dotenv: installed


## Step 2: Load and Validate Configuration

In [4]:
# Load configuration
from config import Config

print("\n" + "="*60)
print("CONFIGURATION VALIDATION")
print("="*60)

try:
    Config.validate()
    print(f"\n✓ Qdrant URL: {Config.QDRANT_URL}")
    print(f"✓ Collection Name: {Config.QDRANT_COLLECTION_NAME}")
    print(f"✓ Data Folder: {Config.DATA_FOLDER}")
    if Config.AZURE_OPENAI_ENDPOINT:
        print(f"✓ Chat Deployment: {Config.AZURE_OPENAI_CHAT_DEPLOYMENT}")
        print(f"✓ Embedding Deployment: {Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT}")
        print(f"✓ Embedding Dimensions: {Config.EMBEDDING_DIMENSIONS}")
    else:
        print(f"✓ OpenAI Model: {Config.OPENAI_MODEL}")
        print(f"✓ Embedding Model: {Config.EMBEDDING_MODEL}")
    print(f"\n✅ Configuration Valid!")
except Exception as e:
    print(f"\n❌ Configuration Error: {e}")



CONFIGURATION VALIDATION

VALIDATING CONFIGURATION
✓ Azure OpenAI Endpoint: https://initiative.openai.azure.com/
✓ Azure API Version: 2025-01-01-preview
✓ Chat Deployment: gpt-4.1
✓ Embedding Deployment: text-embedding-3-large
✓ Mode: Azure OpenAI
✓ Qdrant URL: http://localhost:6333
✓ Project root: c:\RAG
✓ Data folder: c:\RAG\data
✓ Notebooks folder: c:\RAG\notebooks
✓ Embedding Dimensions: 3072

✅ Configuration validated successfully!


✓ Qdrant URL: http://localhost:6333
✓ Collection Name: confidential_docs
✓ Data Folder: c:\RAG\data
✓ Chat Deployment: gpt-4.1
✓ Embedding Deployment: text-embedding-3-large
✓ Embedding Dimensions: 3072

✅ Configuration Valid!


## Step 3: Test Qdrant Connection

In [5]:
# Test Qdrant connection
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

print("\n" + "="*60)
print("QDRANT CONNECTION TEST")
print("="*60)

try:
    # Connect to Qdrant
    client = QdrantClient(url=Config.QDRANT_URL)
    
    # Get collections
    collections = client.get_collections()
    print(f"\n✓ Connected to Qdrant at {Config.QDRANT_URL}")
    print(f"✓ Existing collections: {len(collections.collections)}")
    
    for collection in collections.collections:
        print(f"  - {collection.name}")
    
    print(f"\n✅ Qdrant is running and accessible!")
    
except Exception as e:
    print(f"\n❌ Qdrant Connection Error: {e}")
    print(f"\n💡 Make sure Qdrant is running:")
    print(f"   docker run -d -p 6333:6333 -v ./qdrant_storage:/qdrant/storage qdrant/qdrant:latest")


QDRANT CONNECTION TEST

✓ Connected to Qdrant at http://localhost:6333
✓ Existing collections: 1
  - confidential_docs

✅ Qdrant is running and accessible!


## Step 4: Test OpenAI API Connection

In [6]:
# Test Azure OpenAI / OpenAI API connection
from openai import OpenAI, AzureOpenAI

print("\n" + "="*60)
print("OPENAI API TEST")
print("="*60)

try:
    if Config.AZURE_OPENAI_ENDPOINT:
        client_openai = AzureOpenAI(
            api_key=Config.AZURE_OPENAI_API_KEY,
            azure_endpoint=Config.AZURE_OPENAI_ENDPOINT,
            api_version=Config.AZURE_OPENAI_API_VERSION
        )
        embedding_model = Config.AZURE_OPENAI_EMBEDDING_DEPLOYMENT
        mode = "Azure OpenAI"
    else:
        client_openai = OpenAI(api_key=Config.OPENAI_API_KEY)
        embedding_model = Config.EMBEDDING_MODEL
        mode = "Standard OpenAI"

    # Test with a simple embedding
    response = client_openai.embeddings.create(
        model=embedding_model,
        input="Test connection"
    )

    embedding_dim = len(response.data[0].embedding)
    print(f"\n✓ Mode: {mode}")
    print(f"✓ Embedding Deployment/Model: {embedding_model}")
    print(f"✓ Embedding Dimensions: {embedding_dim}")
    print(f"\n✅ API is working!")

except Exception as e:
    print(f"\n❌ API Error: {e}")
    print(f"\n💡 Check your AZURE_OPENAI_API_KEY and AZURE_OPENAI_ENDPOINT in .env file")



OPENAI API TEST

✓ Mode: Azure OpenAI
✓ Embedding Deployment/Model: text-embedding-3-large
✓ Embedding Dimensions: 3072

✅ API is working!


## Step 5: Verify Folder Structure

In [7]:
# Check folder structure
print("\n" + "="*60)
print("FOLDER STRUCTURE VERIFICATION")
print("="*60)

folders_to_check = [
    Config.DATA_FOLDER,
    Config.NOTEBOOKS_FOLDER,
    Config.QDRANT_STORAGE,
    Config.PROJECT_ROOT / "src"
]

print()
for folder in folders_to_check:
    if folder.exists():
        print(f"✓ {folder.name:20s} exists at {folder}")
    else:
        print(f"⚠ {folder.name:20s} missing - creating...")
        folder.mkdir(parents=True, exist_ok=True)
        print(f"  ✓ Created: {folder}")

print(f"\n✅ All folders ready!")


FOLDER STRUCTURE VERIFICATION

✓ data                 exists at c:\RAG\data
✓ notebooks            exists at c:\RAG\notebooks
✓ qdrant_storage       exists at c:\RAG\qdrant_storage
✓ src                  exists at c:\RAG\src

✅ All folders ready!


## Step 6: Test Multi-File Loader

In [8]:
# Test the multi-file loader
from multi_file_loader import MultiFileLoader

print("\n" + "="*60)
print("MULTI-FILE LOADER TEST")
print("="*60 + "\n")

loader = MultiFileLoader()
print(f"\n✅ Multi-File Loader initialized successfully!")


MULTI-FILE LOADER TEST


✅ Multi-File Loader initialized successfully!


## Summary: Stage 1 Complete ✅

### What We Verified:
- ✅ Python environment working
- ✅ All dependencies installed
- ✅ Qdrant running and accessible
- ✅ OpenAI API working
- ✅ Folder structure created
- ✅ Configuration loaded
- ✅ Multi-file loader initialized

### Next Steps:
Continue to **01_file_loading.ipynb** to implement document loading functionality.

In [9]:
# Final status report
print("\n" + "="*60)
print("🎉 STAGE 1: ENVIRONMENT SETUP - COMPLETE!")
print("="*60)
print("\nReady to proceed to Stage 2: File Loading")
print("Next notebook: 01_file_loading.ipynb")


🎉 STAGE 1: ENVIRONMENT SETUP - COMPLETE!

Ready to proceed to Stage 2: File Loading
Next notebook: 01_file_loading.ipynb
